#Classificação de Textos Tóxicos em Português com BERTimbau

Neste notebook vamos treinar e avaliar um modelo baseado no **BERTimbau**  
para detecção de linguagem tóxica usando o dataset **Hatecheck-Portuguese**.

In [1]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [21]:
import torch
import evaluate
import numpy as np
from collections import Counter
from huggingface_hub import login
from torch.nn import CrossEntropyLoss
from datasets import load_dataset, DatasetDict
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments


## 2. Carregar o dataset

O dataset está disponível no HuggingFace:  
https://huggingface.co/datasets/Paul/hatecheck-portuguese  

- Coluna **test_case** → contém os textos  
- Coluna **label_gold** → contém os rótulos (0 = não tóxico, 1 = tóxico)

In [3]:
from datasets import load_dataset
df = load_dataset("Paul/hatecheck-portuguese")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/3691 [00:00<?, ? examples/s]

##3. Separar em treino e teste

Como o dataset vem apenas em um *split* (`train`),  
vamos separar em **80% treino** e **20% teste**.

In [5]:
dataset = df["test"].train_test_split(test_size=0.2, seed=42)
dataset = DatasetDict({
    "train": dataset["train"],
    "test": dataset["test"]
})

In [7]:
# Mudança das entradas da coluna label_gold para uma saída binária

label_map = {"non-hateful": 0, "hateful": 1}
def map_labels(batch):
    batch["label"] = [label_map[label] for label in batch["label_gold"]]
    return batch

dataset = dataset.map(map_labels, batched=True)

Map:   0%|          | 0/2952 [00:00<?, ? examples/s]

Map:   0%|          | 0/739 [00:00<?, ? examples/s]

## 4. Tokenização com BERTimbau

O modelo que vamos usar é o **neuralmind/bert-base-portuguese-cased**.  
Aqui transformamos os textos em *tokens* para que o BERT consiga processá-los.
python
Copiar código

In [8]:
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["test_case"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/2952 [00:00<?, ? examples/s]

Map:   0%|          | 0/739 [00:00<?, ? examples/s]

In [10]:
# Reduzir tamanho para treinar mais rápido
tokenized_datasets["train"] = tokenized_datasets["train"].shuffle(seed=42).select(range(300))
tokenized_datasets["test"] = tokenized_datasets["test"].shuffle(seed=42).select(range(100))

In [13]:
# Verificação de balanceamento

print("Distribuição antes do balanceamento:", Counter(tokenized_datasets["train"]["label"]))

Distribuição antes do balanceamento: Counter({1: 211, 0: 89})


In [16]:
# Calcular pesos para a loss com base na distribuição das classes

labels_train = tokenized_datasets["train"]["label"]
class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=labels_train)
class_weights = torch.tensor(class_weights).float().to("cuda" if torch.cuda.is_available() else "cpu")
print(f"Pesos das classes: {class_weights}")

Pesos das classes: tensor([1.6854, 0.7109], device='cuda:0')


## 5. Preparar o modelo

Carregamos o **BERTimbau** para classificação de sequência.  
Como é binário, usamos **num_labels=2**.

In [17]:
num_labels = 2
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(29794, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## 6. Definir métricas e argumentos de treino

Vamos avaliar com **accuracy** e **F1-score**, pois o dataset pode ser desbalanceado.

In [22]:
# Métricas
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=p.label_ids),
        "f1": f1.compute(predictions=preds, references=p.label_ids, average="weighted")
    }


# Trainer customizado para usar pesos na loss

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


# Argumentos


training_args = TrainingArguments(
    output_dir="./results",
    report_to="none",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=10
)

## 7. Treinamento do modelo

In [23]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights
)

trainer.train()

/tmp/ipython-input-1711000678.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.692100,0.670872,{'accuracy': 0.62},{'f1': 0.6462957937584802}
2,0.639100,0.651622,{'accuracy': 0.72},{'f1': 0.7237333333333333}


TrainOutput(global_step=38, training_loss=0.6636770148026315, metrics={'train_runtime': 28.5643, 'train_samples_per_second': 21.005, 'train_steps_per_second': 1.33, 'total_flos': 39466658304000.0, 'train_loss': 0.6636770148026315, 'epoch': 2.0})

##8. Avaliação no conjunto de teste

In [25]:
results = trainer.evaluate()
results

{'eval_loss': 0.6516222357749939,
 'eval_accuracy': {'accuracy': 0.72},
 'eval_f1': {'f1': 0.7237333333333333},
 'eval_runtime': 0.8435,
 'eval_samples_per_second': 118.554,
 'eval_steps_per_second': 8.299,
 'epoch': 2.0}

##9. Exemplos de classificação

Vamos testar o modelo em alguns exemplos manuais.

In [27]:
texts = [
    "Você é um idiota.",
    "Bom dia, espero que esteja tudo bem.",
    "Esse grupo de pessoas não merece respeito.",
    "Adoro trabalhar com minha equipe."
]

inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=1)

for text, pred in zip(texts, predictions):
    label = "tóxico" if pred.item() == 1 else "não tóxico"
    print(f"\nTexto: {text}\nClassificação: {label}")


Texto: Você é um idiota.
Classificação: não tóxico

Texto: Bom dia, espero que esteja tudo bem.
Classificação: não tóxico

Texto: Esse grupo de pessoas não merece respeito.
Classificação: tóxico

Texto: Adoro trabalhar com minha equipe.
Classificação: não tóxico
